## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits (ya escalados por 1/T) antes de exponenciar, lo que evita desbordamientos. Con los logits (t0 = 2.0, t1 = 1.0, t2 = 0.5, t3 = 0.2, t4 = -1.0, t5 = -3.0), cada distribución suma exactamente 1.0. La entropía se calcula en bits (log base 2) sobre la distribución completa; como los seis tokens tienen probabilidad positiva, no se requiere corrección.

| T | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|---|------|------|------|------|------|------|------|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Nota: para T = 0.5 la probabilidad de t5 es 3.7367e-05, que se muestra como 0.0000 al redondear a cuatro decimales. La tendencia es clara: al bajar la temperatura la masa se concentra en t0 (0.8231 con T = 0.5 frente a 0.3583 con T = 2.0) y la entropía cae de 2.2888 a 0.9147 bits; al subirla, la distribución se aplana y la entropía crece.

In [1]:
# T1: código aprobado por el crítico
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----- Datos del enunciado -----
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


def softmax_con_temperatura(z, T):
    """Softmax con temperatura p_i = exp(z_i/T) / sum_j exp(z_j/T),
    numéricamente estable: se resta el máximo de los logits escalados."""
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    escalados = z / T
    escalados = escalados - escalados.max()  # estabilidad numérica
    e = np.exp(escalados)
    return e / e.sum()


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -sum p_i * log2(p_i)."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


# ----- Cálculo de distribuciones y entropías -----
filas = []
entropias = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    H = entropia_bits(p)
    entropias[T] = H
    fila = {"T": T}
    for tok, pi in zip(tokens, p):
        fila[tok] = round(float(pi), 4)  # cuatro decimales
    fila["entropia_bits"] = round(H, 4)
    filas.append(fila)

tabla = pd.DataFrame(filas)

# ----- Verificaciones del criterio de éxito -----
sumas = [float(np.sum(softmax_con_temperatura(logits, T))) for T in temperaturas]
assert all(abs(s - 1.0) < 1e-12 for s in sumas), "Cada distribución debe sumar 1.0"
Hs = [entropias[T] for T in temperaturas]
assert Hs[0] < Hs[1] < Hs[2], "La entropía debe crecer con T: H(0.5) < H(1) < H(2)"

# ----- resultados.json (contrato de la subtarea) -----
resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": {tok: float(pi) for tok, pi in
                   zip(tokens, softmax_con_temperatura(logits, T))}
        for T in temperaturas
    },
    "sumas_distribuciones": {f"T={T}": s for T, s in zip(temperaturas, sumas)},
    "entropias_bits": {f"T={T}": H for T, H in zip(temperaturas, Hs)},
    "tabla_4_decimales": tabla.to_dict(orient="records"),
    "supuestos": [
        "La entropía se calcula en bits (log2) sobre la distribución completa; "
        "los seis tokens tienen probabilidad positiva, no se requiere corrección.",
        "La estabilidad numérica se logra restando el máximo de los logits ya "
        "escalados por 1/T antes de la exponenciación.",
    ],
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----- Figura: efecto de la temperatura -----
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tokens))
ancho = 0.25
for i, T in enumerate(temperaturas):
    p = softmax_con_temperatura(logits, T)
    ax.bar(x + (i - 1) * ancho, p, width=ancho, label=f"T={T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura (logits 2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
fig.tight_layout()
fig.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----- Resumen -----
print("Tabla (4 decimales):")
print(tabla.to_string(index=False))
print("\nEntropías en bits:")
for T in temperaturas:
    print(f"  T={T}: {entropias[T]:.4f}")
print("\nCada distribución suma 1.0 y H(0.5) < H(1) < H(2): verificado.")
print("Archivos escritos: resultados.json, t1_softmax_temperatura.png")


Tabla (4 decimales):
  T     t0     t1     t2     t3     t4     t5  entropia_bits
0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000         0.9147
1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037         1.7998
2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294         2.2888

Entropías en bits:
  T=0.5: 0.9147
  T=1.0: 1.7998
  T=2.0: 2.2888

Cada distribución suma 1.0 y H(0.5) < H(1) < H(2): verificado.
Archivos escritos: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p, p = 0.9)

Sobre la distribución de T = 1 de la Parte 1, se ordenan los tokens por probabilidad descendente y se conserva el conjunto mínimo cuya probabilidad acumulada alcanza o supera p = 0.9. Tras t2 la acumulada es 0.8776 (aún por debajo de 0.9), por lo que debe incluirse t3, con el que la acumulada llega a 0.9688. Sobreviven t0, t1, t2 y t3 (4 tokens), con una masa de núcleo de 0.9688; se descartan t4 y t5, con una masa descartada de 0.0312. La renormalización divide cada probabilidad conservada entre la masa del núcleo, de modo que las probabilidades resultantes suman exactamente 1.0.

| Token | p (T = 1) | Acumulada | ¿Conservado? | p renormalizada |
|-------|------|------|------|------|
| t0 | 0.5516 | 0.5516 | Sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | Sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | Sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | Sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | No | — |
| t5 | 0.0037 | 1.0000 | No | — |

El efecto de top-p es recortar la cola de baja probabilidad y redistribuir su masa entre los tokens del núcleo: t0 pasa de 0.5516 a 0.5694 y t3 de 0.0912 a 0.0941, mientras el orden relativo se mantiene.

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# T2: Top-p (nucleus) con p = 0.9 sobre la distribución de T = 1
# La distribución de T=1 se CARGA de la subtarea previa (entrada/T1/),
# no se recalcula.
# ------------------------------------------------------------------

with open("entrada/T1/resultados.json", "r", encoding="utf-8") as f:
    t1_res = json.load(f)

tokens = list(t1_res["tokens"])
probs = np.array([t1_res["distribuciones"]["T=1.0"][t] for t in tokens], dtype=float)

# 1) Ordenar tokens por probabilidad descendente
orden = np.argsort(-probs)
tokens_ord = [tokens[i] for i in orden]
probs_ord = probs[orden]
acum = np.cumsum(probs_ord)

# 2) Conjunto mínimo cuya probabilidad acumulada alcanza p = 0.9
p = 0.9
idx = int(np.searchsorted(acum, p))   # primer índice con acum >= p
k = min(idx + 1, len(tokens))         # número de tokens conservados
sobreviven = tokens_ord[:k]
descartados = tokens_ord[k:]
acum_nucleo = float(acum[k - 1])
probs_nucleo = probs_ord[:k]

# 3) Renormalizar las probabilidades del núcleo
probs_renorm = probs_nucleo / probs_nucleo.sum()

# Verificaciones (garantizadas por la construcción anterior)
assert acum_nucleo >= p, "la masa del núcleo debe alcanzar p"
if k > 1:
    assert acum[k - 2] < p, "minimalidad: sin el último token no se alcanza p"
assert abs(probs_renorm.sum() - 1.0) < 1e-12, "la renormalizada debe sumar 1"

# 4) resultados.json (contrato de la subtarea)
resultados = {
    "p": p,
    "temperatura": 1.0,
    "distribucion_T1_ordenada": {t: float(pr) for t, pr in zip(tokens_ord, probs_ord)},
    "probabilidades_acumuladas_ordenadas": {t: float(a) for t, a in zip(tokens_ord, acum)},
    "tokens_sobrevivientes": sobreviven,
    "num_tokens_conservados": k,
    "probabilidad_acumulada_nucleo": acum_nucleo,
    "distribucion_renormalizada": {t: float(pr) for t, pr in zip(sobreviven, probs_renorm)},
    "suma_distribucion_renormalizada": float(probs_renorm.sum()),
    "tokens_descartados": descartados,
    "masa_descartada": float(probs_ord[k:].sum()),
    "tabla": [
        {
            "token": t,
            "prob_original_T1": float(pr),
            "prob_acumulada": float(a),
            "conservado": bool(i < k),
            "prob_renormalizada": float(probs_renorm[i]) if i < k else None,
        }
        for i, (t, pr, a) in enumerate(zip(tokens_ord, probs_ord, acum))
    ],
    "supuestos": [
        "El conjunto mínimo se define incluyendo el primer token (en orden descendente) cuya "
        "probabilidad acumulada alcanza o supera p=0.9; los tokens posteriores se descartan, "
        "aunque la masa del núcleo quede por encima de 0.9.",
        "La distribución de T=1 se carga de entrada/T1/resultados.json (subtarea T1) y no se recalcula.",
        "La renormalización divide cada probabilidad conservada entre la masa total del núcleo, "
        "de modo que las probabilidades resultantes suman exactamente 1.0.",
    ],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# 5) Figura: distribución original vs. renormalizada tras el corte top-p
renorm_full = np.zeros(len(tokens))
for t, pr in zip(sobreviven, probs_renorm):
    renorm_full[tokens.index(t)] = pr

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(tokens))
ax.bar(x - 0.2, probs, width=0.4, label="T=1 original")
ax.bar(x + 0.2, renorm_full, width=0.4, label="Top-p (p=0.9) renormalizada")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_title("Muestreo de núcleo top-p (p=0.9) sobre la distribución de T=1")
ax.legend()
fig.tight_layout()
fig.savefig("top_p_T1.png", dpi=120)
plt.close(fig)

# 6) Resumen breve
print("Distribución T=1 ordenada (desc):",
      {t: round(float(pr), 4) for t, pr in zip(tokens_ord, probs_ord)})
print("Probabilidades acumuladas:", [round(float(a), 4) for a in acum])
print(f"Tokens que sobreviven al corte top-p (p={p}): {sobreviven}")
print(f"Probabilidad acumulada del núcleo: {acum_nucleo:.6f} (>= {p})")
print("Distribución renormalizada:",
      {t: round(float(pr), 6) for t, pr in zip(sobreviven, probs_renorm)})
print(f"Suma de la distribución renormalizada: {float(probs_renorm.sum()):.12f}")
print("Tokens descartados:", descartados,
      "| masa descartada:", round(float(probs_ord[k:].sum()), 6))
print("Resultados escritos en resultados.json; figura en top_p_T1.png")


Distribución T=1 ordenada (desc): {'t0': 0.5516, 't1': 0.2029, 't2': 0.1231, 't3': 0.0912, 't4': 0.0275, 't5': 0.0037}
Probabilidades acumuladas: [0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0]
Tokens que sobreviven al corte top-p (p=0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.968820 (>= 0.9)
Distribución renormalizada: {'t0': 0.569376, 't1': 0.209462, 't2': 0.127045, 't3': 0.094117}
Suma de la distribución renormalizada: 1.000000000000
Tokens descartados: ['t4', 't5'] | masa descartada: 0.03118
Resultados escritos en resultados.json; figura en top_p_T1.png


## Parte 3 — Comprobación empírica

Se toman 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0), muestreando sobre los tokens sobrevivientes del núcleo (t0–t3). Las frecuencias observadas son los conteos divididos entre 10 000, directamente comparables con las probabilidades teóricas en el gráfico de barras generado por la celda.

| Token | Probabilidad teórica | Conteo observado | Frecuencia observada |
|-------|------|------|------|
| t0 | 0.5694 | 5682 | 0.5682 |
| t1 | 0.2095 | 2093 | 0.2093 |
| t2 | 0.1270 | 1296 | 0.1296 |
| t3 | 0.0941 | 929 | 0.0929 |

La divergencia KL de la distribución empírica respecto de la teórica, KL(empírica ‖ teórica) = Σ q_i log2(q_i / p_i) en bits, es 5.0069e-05 bits. Este valor, prácticamente cero, confirma que las frecuencias observadas reproducen las probabilidades teóricas: las mayores desviaciones absolutas (t2 con 0.1296 frente a 0.1270 y t3 con 0.0929 frente a 0.0941) son del orden del 0.002 y son consistentes con el ruido de muestreo esperable con n = 10 000.

In [3]:
# T3: código aprobado por el crítico
# SUBTAREA T3: Comprobación empírica de la distribución renormalizada (top-p, p=0.9, T=1)
# - 10 000 muestras con numpy.random.default_rng(0)
# - Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# - Divergencia KL (empírica || teórica) en bits

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1) Cargar la distribución renormalizada producida por T2
# ------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        res_t2 = json.load(f)
    tokens = list(res_t2["distribucion_renormalizada"].keys())
    p_teorica = np.array([res_t2["distribucion_renormalizada"][t] for t in tokens],
                         dtype=float)
else:
    # Fallback defensivo: reconstruir desde los logits del enunciado
    # (solo se usa si entrada/T2/resultados.json no existe)
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
    tokens_todos = [f"t{i}" for i in range(6)]
    z = logits - logits.max()
    p_t1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_t1)
    acum = np.cumsum(p_t1[orden])
    n_nucleo = int(np.searchsorted(acum, 0.9) + 1)
    idx_nucleo = np.sort(orden[:n_nucleo])
    tokens = [tokens_todos[i] for i in idx_nucleo]
    p_teorica = p_t1[idx_nucleo] / p_t1[idx_nucleo].sum()

p_teorica = p_teorica / p_teorica.sum()  # garantiza suma exactamente 1
n_tokens = len(tokens)

# ------------------------------------------------------------------
# 2) Muestrear 10 000 veces con la semilla exigida
# ------------------------------------------------------------------
N = 10_000
rng = np.random.default_rng(0)
indices = np.arange(n_tokens)
muestras = rng.choice(indices, size=N, p=p_teorica)

conteos = np.bincount(muestras, minlength=n_tokens)
frecuencias_obs = conteos / N

# ------------------------------------------------------------------
# 3) Divergencia KL(empírica || teórica) en bits
#    KL = Σ q_i * log2(q_i / p_i); términos con q_i = 0 valen 0
# ------------------------------------------------------------------
mascara = frecuencias_obs > 0
kl_bits = float(np.sum(frecuencias_obs[mascara] *
                       np.log2(frecuencias_obs[mascara] / p_teorica[mascara])))

# ------------------------------------------------------------------
# 4) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# ------------------------------------------------------------------
x = np.arange(n_tokens)
ancho = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - ancho / 2, frecuencias_obs, width=ancho,
       label="Frecuencia observada (10 000 muestras)", color="#4C72B0")
ax.bar(x + ancho / 2, p_teorica, width=ancho,
       label="Probabilidad teórica (renormalizada)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title("Parte 3: frecuencias observadas vs. probabilidades teóricas\n"
             "muestreo top-p (p=0.9, T=1), 10 000 muestras, rng(0)")
ax.legend()
for i, (fo, pt) in enumerate(zip(frecuencias_obs, p_teorica)):
    ax.text(i - ancho / 2, fo + 0.005, f"{fo:.4f}", ha="center", fontsize=8)
    ax.text(i + ancho / 2, pt + 0.005, f"{pt:.4f}", ha="center", fontsize=8)
ax.set_ylim(0, max(p_teorica.max(), frecuencias_obs.max()) * 1.15)
fig.tight_layout()
plt.savefig("parte3_frecuencias_vs_teorica.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5) Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "n_muestras": N,
    "semilla": 0,
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(tokens, frecuencias_obs)},
    "kl_bits_empirica_vs_teorica": kl_bits,
    "figura": "parte3_frecuencias_vs_teorica.png",
    "supuestos": [
        "La distribución renormalizada se carga de entrada/T2/resultados.json (subtarea T2); "
        "el muestreo se realiza sobre los tokens sobrevivientes del núcleo (t0-t3) con sus "
        "probabilidades renormalizadas.",
        "La divergencia KL se calcula como KL(empírica || teórica) = Σ q_i log2(q_i / p_i) "
        "sobre los tokens del núcleo, en bits (log base 2).",
        "Las 'frecuencias observadas' son conteos divididos entre 10 000, de modo que son "
        "directamente comparables con las probabilidades teóricas en el gráfico de barras.",
    ],
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# 6) Resumen breve
# ------------------------------------------------------------------
print("=== SUBTAREA T3: comprobación empírica (top-p p=0.9, T=1) ===")
print(f"Muestras: {N} | Semilla: 0 | Tokens del núcleo: {tokens}")
for t, c, fo, pt in zip(tokens, conteos, frecuencias_obs, p_teorica):
    print(f"  {t}: conteo={c:6d}  freq_obs={fo:.6f}  p_teorica={pt:.6f}")
print(f"Divergencia KL(empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: parte3_frecuencias_vs_teorica.png")
print("Resultados escritos en resultados.json")


=== SUBTAREA T3: comprobación empírica (top-p p=0.9, T=1) ===
Muestras: 10000 | Semilla: 0 | Tokens del núcleo: ['t0', 't1', 't2', 't3']
  t0: conteo=  5682  freq_obs=0.568200  p_teorica=0.569376
  t1: conteo=  2093  freq_obs=0.209300  p_teorica=0.209462
  t2: conteo=  1296  freq_obs=0.129600  p_teorica=0.127045
  t3: conteo=   929  freq_obs=0.092900  p_teorica=0.094117
Divergencia KL(empírica || teórica) = 0.000050 bits
Figura guardada: parte3_frecuencias_vs_teorica.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0.** Dividir los logits por una temperatura muy pequeña amplifica sus diferencias: el cociente exp(z_i/T) entre el token de mayor logit y cualquier otro crece sin acotarse, de modo que toda la masa se concentra en el máximo, t0 (logit 2.0), y la entropía tiende a 0. Las entropías de la Parte 1 muestran esa trayectoria: al reducir T a la mitad (de 1.0 a 0.5) la entropía cae de 1.7998 a 0.9147 bits, menos de la mitad, y t0 ya concentra 0.8231 de la masa. Extrapolando la tendencia, en el límite T → 0 la distribución es una masa puntual sobre t0 con entropía 0.

**Cuando T → ∞.** Dividir por una temperatura muy grande hace z_i/T → 0 para todo token, de modo que exp(z_i/T) → 1 y la distribución tiende a la uniforme sobre los seis tokens, con entropía límite log2(6) bits, el máximo posible para un vocabulario de seis. La evidencia apunta en esa dirección: al subir T de 1.0 a 2.0 la entropía crece de 1.7998 a 2.2888 bits, acercándose a log2(6), mientras las probabilidades se aplanan (t0 baja de 0.5516 a 0.3583 y t5 sube de 0.0037 a 0.0294).

**Por qué T → 0 equivale a la decodificación voraz.** La decodificación voraz (greedy) elige en cada paso el token de mayor logit —aquí t0— de forma determinista. El muestreo con temperatura T → 0 asigna probabilidad que tiende a 1 precisamente a ese token, de modo que el muestreador devuelve t0 casi con seguridad: la misma decisión que greedy. Una entropía de 0 bits caracteriza una distribución determinista, idéntica en la práctica a la selección argmax de greedy. La temperatura interpola así entre dos extremos: greedy determinista (T → 0, entropía 0) y sorteo uniforme (T → ∞, entropía log2(6)), con las entropías medidas 0.9147, 1.7998 y 2.2888 bits marcando puntos intermedios de ese recorrido.